# Augmentation Sweeps für ResNet18, 34, 50
finde die optimalen Augmentation-Parameter für alle drei Modelle

#### Mount MyDrive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Clone Repository

In [ ]:
# Clone repository (adjust branch name if needed)
!git clone -b feat/augment-sweep --single-branch https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

#### Copy Dataset

In [ ]:
from tqdm import tqdm

# Copy dataset from Drive to local VM
!cp /content/drive/MyDrive/ImageNetSubset.zip /content/

# Unzip to /content/
!unzip -q /content/ImageNetSubset.zip -d /content/

# Move to project datasets folder
!mv /content/ImageNetSubset /content/xAI-proj-m-ws2526/datasets

# Remove zip to save space
!rm /content/ImageNetSubset.zip

print("✓ Dataset copied and extracted")

#### Setup Project Root

In [ ]:
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward to find the outermost folder containing common project markers."""
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent
    return root or start

# Get cloned repository path
cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

# Change into repository if exists
if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

# Find and set project root
ROOT = find_project_root(Path.cwd()).resolve()
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Working directory: {Path.cwd()}")
print(f"Root on sys.path: {str(ROOT) in sys.path}")

# Set data directory environment variable
os.environ['DATA_DIR'] = str(ROOT / "datasets")
print(f"Data directory: {os.environ['DATA_DIR']}")

#### GPU Check

In [ ]:
import torch
print(f"\n{'='*60}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print('='*60)

#### Requirements

In [ ]:
# Install dependencies
!pip install -r experiments/bagging/requirements.txt --quiet

# Additional packages for sweeps
!pip install -q wandb pyyaml tqdm

#### WandB Login

In [ ]:
import wandb
wandb.login()

# Setzen Sie hier Ihren eigenen Projektnamen
PROJECT_NAME = "ihr-projekt-name"  # z.B. "imagenet-augmentation" oder "resnet-sweep"
print(f"✓ Using project: {PROJECT_NAME}")

#### Verify Project Structure

In [ ]:
# Verify all necessary files exist
print("\n" + "="*60)
print("Project Structure Check")
print("="*60)

files_to_check = [
    "experiments/bagging/src/data/dataset.py",
    "experiments/bagging/src/training/train_augmentation_sweep.py",
    "experiments/bagging/configs/sweep_resnet18_augment.yaml",
    "experiments/bagging/configs/sweep_resnet34_augment.yaml",
    "experiments/bagging/configs/sweep_resnet50_augment.yaml",
]

all_exist = True
for file in files_to_check:
    exists = Path(file).exists()
    status = "✓" if exists else "✗"
    print(f"{status} {file}")
    if not exists:
        all_exist = False

if all_exist:
    print("\n✓ All required files found!")
else:
    print("\n✗ Some files are missing. Please check your repository.")

#### Sweep Configurations

In [ ]:
import yaml

# Load sweep configurations
sweep_configs = {}

for model in ['resnet18', 'resnet34', 'resnet50']:
    config_path = f'experiments/bagging/configs/sweep_{model}_augment.yaml'
    
    if Path(config_path).exists():
        with open(config_path, 'r') as f:
            sweep_configs[model] = yaml.safe_load(f)
        print(f"✓ Loaded config for {model}")
    else:
        print(f"✗ Config not found: {config_path}")

print(f"\n✓ {len(sweep_configs)} sweep configurations loaded")

#### Training Function

In [ ]:
# Import training modules
sys.path.insert(0, str(ROOT / "experiments" / "bagging"))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torchvision import models
from tqdm import tqdm
from src.data.dataset import create_data_loaders


def get_model(model_name, num_classes):
    """Create pretrained model."""
    model_dict = {
        'resnet18': models.resnet18,
        'resnet34': models.resnet34,
        'resnet50': models.resnet50,
    }
    model = model_dict[model_name](pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def extract_aug_params(config, model_name):
    """Extract augmentation parameters based on model."""
    aug_params = {}
    
    if model_name == "resnet18":
        aug_params = {
            'brightness': config.get('brightness', 0.3),
            'contrast': config.get('contrast', 0.3),
            'saturation': config.get('saturation', 0.3),
            'hue': config.get('hue', 0.1),
            'rotation': config.get('rotation', 15),
        }
    elif model_name == "resnet34":
        aug_params = {
            'policy': config.get('policy', 'IMAGENET'),
            'use_trivial_augment': config.get('use_trivial_augment', False),
        }
    elif model_name == "resnet50":
        aug_params = {
            'num_ops': config.get('num_ops', 2),
            'magnitude': config.get('magnitude', 9),
            'use_cutout': config.get('use_cutout', True),
        }
    
    return aug_params


def train_epoch(model, loader, criterion, optimizer, device, scaler):
    """Train one epoch with mixed precision."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training", leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in tqdm(loader, desc="Validation", leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return running_loss / total, 100.0 * correct / total

# Vollständige Training-Funktion integriert
def train():
    """Main training function for WandB sweep."""
    wandb.init()
    config = wandb.config
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_name = config.model_name
    aug_params = extract_aug_params(config, model_name)
    
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"Augmentation: {aug_params}")
    print('='*60)
    
    # Create data loaders
    data_dir = os.environ.get('DATA_DIR')
    train_loader, val_loader, num_classes = create_data_loaders(
        data_dir=data_dir,
        batch_size=config.batch_size,
        model_name=model_name,
        aug_params=aug_params,
        num_workers=2,
        pin_memory=True,
    )
    
    print(f"Dataset: {num_classes} classes")
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    
    # Setup model and training - GLEICH WIE IN trainer.py
    model = get_model(model_name, num_classes).to(device)
    
    # ← GLEICH wie in trainer.py
    criterion = nn.CrossEntropyLoss(
        label_smoothing=config.get('label_smoothing', 0.1)
    )
    
    # ← GLEICH wie in trainer.py: SGD mit fixen Defaults
    optimizer = optim.SGD(
        model.parameters(),
        lr=config.learning_rate,  # Aus Sweep
        momentum=config.get('momentum', 0.9),  # Default 0.9
        weight_decay=config.get('weight_decay', 1e-4)  # Default 1e-4 = 0.0001
    )
    
    # Scheduler anpassen - Verwenden Sie StepLR wie in trainer.py oder OneCycleLR
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=config.learning_rate,
        epochs=config.epochs,
        steps_per_epoch=len(train_loader)
    )
    
    scaler = GradScaler()  # Mixed precision behalten
    
    # Training loop
    best_val_acc = 0.0
    
    for epoch in range(config.epochs):
        print(f"\nEpoch {epoch+1}/{config.epochs}")
        print("-" * 60)
        
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, scaler
        )
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # Log to WandB
        wandb.log({
            'epoch': epoch,
            'train/loss': train_loss,
            'train/accuracy': train_acc,
            'val/loss': val_loss,
            'val/accuracy': val_acc,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            wandb.run.summary['best_val_accuracy'] = best_val_acc
        
        scheduler.step()
        
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}% (Best: {best_val_acc:.2f}%)")
    
    wandb.finish()

print("✓ Training function defined")

#### Run All Sweeps Automatisch

In [ ]:
import time

# Configuration
RUNS_PER_MODEL = 30
MODELS = ['resnet18', 'resnet34', 'resnet50']
PROJECT_NAME = "restnets-aug"  # projektname for WandB

results = {}

print("\n" + "="*70)
print("STARTING AUGMENTATION SWEEPS FOR ALL MODELS")
print("="*70)
print(f"Project: {PROJECT_NAME}")
print(f"Runs per model: {RUNS_PER_MODEL}")
print(f"Total runs: {RUNS_PER_MODEL * len(MODELS)}")
print("="*70 + "\n")

for model_name in MODELS:
    print("\n" + "="*70)
    print(f"Starting sweep for {model_name.upper()}")
    print("="*70)
    
    # Create sweep
    sweep_id = wandb.sweep(
        sweep=sweep_configs[model_name],
        project=PROJECT_NAME  # take my project name
    )
    
    print(f"✓ Sweep created")
    print(f"  ID: {sweep_id}")
    print(f"  Project: {PROJECT_NAME}")
    print(f"  Model: {model_name}")
    print(f"  Runs: {RUNS_PER_MODEL}")
    print(f"\nStarting agent...\n")
    
    # Run sweep
    start_time = time.time()
    wandb.agent(sweep_id, function=train, count=RUNS_PER_MODEL) # Starte Agent mit FIXED COUNT
    elapsed = time.time() - start_time
    
    results[model_name] = {
        'sweep_id': sweep_id,
        'time': elapsed,
        'runs': RUNS_PER_MODEL
    }
    
    print(f"\n✓ {model_name} completed in {elapsed/60:.1f} minutes")
    print(f"  Average time per run: {elapsed/RUNS_PER_MODEL:.1f} seconds")

print("\n" + "="*70)
print("ALL SWEEPS COMPLETED! 🎉")
print("="*70)

total_time = sum(r['time'] for r in results.values())
print(f"\nTotal time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print("\nResults:")
for model, data in results.items():
    print(f"  {model}:")
    print(f"    Time: {data['time']/60:.1f} min")
    print(f"    Sweep ID: {data['sweep_id']}")
    print(f"    Runs: {data['runs']}")

#### Analyse der besten Ergebnisse

In [ ]:
import pandas as pd
import json

print("\n" + "="*70)
print("ANALYZING BEST RESULTS")
print("="*70)

api = wandb.Api()
best_configs = {}

for model_name in MODELS:
    print(f"\nAnalyzing results for {model_name.upper()}")  # ← FIX
    print("-" * 60)
    
    try:
        # Load runs from THE SAME project
        runs = api.runs(PROJECT_NAME)  # ← FIX: Korrekter Projekt-Name
        model_runs = [r for r in runs if r.config.get('model_name') == model_name]
        
        # Sort by best validation accuracy
        sorted_runs = sorted(
            model_runs,
            key=lambda r: r.summary.get('best_val_accuracy', 0),
            reverse=True
        )
        
        # Display top 3
        for i, run in enumerate(sorted_runs[:3], 1):
            acc = run.summary.get('best_val_accuracy', 0)
            print(f"\n{i}. Run: {run.name}")
            print(f"   Accuracy: {acc:.2f}%")
            print(f"   Config:")
            for k, v in run.config.items():
                if not k.startswith('_'):
                    print(f"     {k}: {v}")
        
        # Store best run
        if sorted_runs:
            best_run = sorted_runs[0]
            best_configs[model_name] = {
                'accuracy': best_run.summary.get('best_val_accuracy', 0),
                'config': {k: v for k, v in best_run.config.items() if not k.startswith('_')},
                'run_name': best_run.name,
                'run_id': best_run.id,
            }
        else:
            print(f"   ⚠️ No runs found for {model_name}")
    
    except Exception as e:
        print(f"Error loading results for {model_name}: {e}")

# Save best configurations
output_file = 'experiments/bagging/configs/best_augmentation_configs.json'
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with open(output_file, 'w') as f:
    json.dump(best_configs, f, indent=2)

print(f"\n{'='*70}")
print(f"✓ Best configurations saved to: {output_file}")
print('='*70)

# Summary
print("\n" + "="*70)
print("BEST CONFIGURATION SUMMARY")
print("="*70)
for model, data in best_configs.items():
    print(f"\n{model.upper()}:")
    print(f"  Best Accuracy: {data['accuracy']:.2f}%")
    print(f"  Run: {data['run_name']}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Creating visualizations...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, model_name in enumerate(MODELS):
    try:
        # Load runs from THE SAME project
        runs = api.runs(PROJECT_NAME)  # ← FIX
        model_runs = [r for r in runs if r.config.get('model_name') == model_name]
        accuracies = [r.summary.get('best_val_accuracy', 0) for r in model_runs if r.summary.get('best_val_accuracy', 0) > 0]
        
        if accuracies:
            axes[idx].hist(accuracies, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
            axes[idx].axvline(max(accuracies), color='red', linestyle='--', 
                            linewidth=2, label=f'Best: {max(accuracies):.2f}%')
            axes[idx].set_xlabel('Validation Accuracy (%)', fontsize=12)
            axes[idx].set_ylabel('Count', fontsize=12)
            axes[idx].set_title(f'{model_name.upper()} Sweep Results', fontsize=14, fontweight='bold')
            axes[idx].legend(fontsize=11)
            axes[idx].grid(alpha=0.3, linestyle='--')
        else:
            axes[idx].text(0.5, 0.5, f'No data for {model_name}', 
                          ha='center', va='center', fontsize=12)
    
    except Exception as e:
        print(f"Error plotting {model_name}: {e}")

plt.tight_layout()
plt.savefig('augmentation_sweep_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved as 'augmentation_sweep_results.png'")